# Consultando Dados: select(), Filtros e Paginação

---

## Contexto: o time de analytics quer os dados

O nosso banco está já está populado, agora o time de vendas pediu alguns relatórios:
- "Lista os 5 produtos mais caros da categoria Eletrônicos"
- "Quais clientes são de SP ou RJ?"
- "Preciso paginar os pedidos em 20 por página"

Nesta aula você vai:
- Usar o `select()` moderno do SQLAlchemy 2.x
- Escolher o formato de retorno certo para cada situação
- Filtrar com `where()`, `and_()`, `or_()` e `like()`
- Ordenar e paginar resultados
- Usar IA para transformar perguntas em queries

---

## 1. Configuração

In [8]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


In [9]:
# verificar tabelas existentes
insp = inspect(engine)
print("Tabelas existentes:", insp.get_table_names())

Tabelas existentes: ['tb_clientes', 'tb_itens_pedidos', 'tb_pedidos', 'tb_produtos']


## 2. O básico do select()

No SQLAlchemy 2.x, toda consulta começa com `select()`.  
Importante: o `select()` apenas **monta** a instrução, ele não vai ao banco ainda.

```
stmt = select(...)           → monta a query (em memória)
session.execute(stmt)        → executa e retorna linhas
session.scalars(stmt)        → executa e retorna objetos ORM
```

### 2.1 Retornando objetos ORM com `scalars()`

Use quando quiser trabalhar com os **objetos completos** acessar atributos, navegar relacionamentos:

In [12]:
with Session(engine) as session:  # Abre uma sessão para interagir com o banco de dados
    clientes = session.scalars(select(Cliente)).all()  # Seleciona todos os registros da tabela Cliente como objetos ORM e os carrega na memória

    print(f"Total de clientes: {len(clientes)}")  # Imprime o número total de clientes encontrados
    for c in clientes:  # Itera objetos Cliente para exibir detalhes de cada um
        print(f"  {c.id} | {c.nome} | {c.cidade}/{c.estado}")  # Exibe o ID, nome e cidade/estado de cada cliente formatado

Total de clientes: 21
  1 | Deborah Foroni | São Paulo/SP
  2 | Srta. Julia Cassiano | São Paulo/SP
  3 | Mathias Casa Grande | Salvador/BA
  4 | Maria Sophia Guerra | Rio de Janeiro/RJ
  5 | Stella Guerra | Brasília/DF
  6 | Alícia Alves | Duque de Caxias/RJ
  7 | Dr. Bryan Carvalho | Guarulhos/SP
  8 | Melissa Campos | Belo Horizonte/MG
  9 | Théo Cirino | Fortaleza/CE
  10 | Luiz Miguel Fogaça | Goiânia/GO
  11 | Dra. Evelyn Correia | Belo Horizonte/MG
  12 | Sr. Mathias Guerra | São Gonçalo/RJ
  13 | Yasmin Fonseca | São Gonçalo/RJ
  14 | Joaquim Oliveira | São Luís/MA
  15 | Pietro Porto | Porto Alegre/RS
  16 | Dra. Marina Borges | Curitiba/PR
  17 | Laís Santos | Campinas/SP
  18 | Dr. Diogo Sousa | Belém/PA
  19 | Anna Liz Sá | Brasília/DF
  20 | Davi Luiz Pimenta | Fortaleza/CE
  21 | Ana Liz Monteiro | Duque de Caxias/RJ


### 2.2 Retornando colunas específicas

Use quando só precisar de alguns campos, trafega menos dado e é mais rápido:

In [14]:
with Session(engine) as session:
    # Seleciona apenas as colunas necessárias
    stmt = select(Cliente.id, Cliente.nome, Cliente.estado)
    linhas = session.execute(stmt).all()

    print("ID  | Nome                 | Estado")
    print("-" * 40)
    for linha in linhas:
        print(f"{linha.id:<4}| {linha.nome:<21}| {linha.estado}")

ID  | Nome                 | Estado
----------------------------------------
1   | Deborah Foroni       | SP
2   | Srta. Julia Cassiano | SP
3   | Mathias Casa Grande  | BA
4   | Maria Sophia Guerra  | RJ
5   | Stella Guerra        | DF
6   | Alícia Alves         | RJ
7   | Dr. Bryan Carvalho   | SP
8   | Melissa Campos       | MG
9   | Théo Cirino          | CE
10  | Luiz Miguel Fogaça   | GO
11  | Dra. Evelyn Correia  | MG
12  | Sr. Mathias Guerra   | RJ
13  | Yasmin Fonseca       | RJ
14  | Joaquim Oliveira     | MA
15  | Pietro Porto         | RS
16  | Dra. Marina Borges   | PR
17  | Laís Santos          | SP
18  | Dr. Diogo Sousa      | PA
19  | Anna Liz Sá          | DF
20  | Davi Luiz Pimenta    | CE
21  | Ana Liz Monteiro     | RJ


### 2.3 Retornando dicionários com `mappings()`

Use quando quiser acessar campos pelo nome — **ótimo para passar para o Pandas**:

In [15]:
with Session(engine) as session:  # Abre uma sessão para executar a query no banco
    stmt = select(Cliente.id, Cliente.nome, Cliente.cidade, Cliente.estado)  # Monta a query para selecionar colunas específicas da tabela Cliente
    dicts = session.execute(stmt).mappings().all()  # Executa a query e retorna os resultados como uma lista de dicionários (mappings)

    # Cada item é como um dicionário
    if dicts:  # Verifica se há resultados (lista não vazia)
        print("Primeiro registro como dicionário:")  # Imprime mensagem se houver dados
        print(dict(dicts[0]))  # Converte o primeiro dicionário para dict (redundante, mas mostra o formato) e imprime

Primeiro registro como dicionário:
{'id': 1, 'nome': 'Deborah Foroni', 'cidade': 'São Paulo', 'estado': 'SP'}


### Resumo: qual método usar?

| O que você quer | Como fazer | Retorna |
|---|---|---|
| Objeto ORM completo | `session.scalars(select(Cliente)).all()` | `[<Cliente>, ...]` |
| Colunas específicas | `session.execute(select(Cliente.id, Cliente.nome)).all()` | `[(1, 'Ana'), ...]` |
| Dicionários | `session.execute(stmt).mappings().all()` | `[{'id': 1, 'nome': 'Ana'}, ...]` |

> ⚠️ **Armadilha:** `scalars()` com múltiplas colunas retorna **apenas a primeira coluna**!  
> Para múltiplas colunas, sempre use `execute().all()`.

---

## 3. Filtrando resultados

### 3.1 Filtro simples com `where()`

In [16]:
with Session(engine) as session:

    # Clientes de SP
    stmt = select(Cliente).where(Cliente.estado == "SP")
    clientes_sp = session.scalars(stmt).all()
    print(f"Clientes em SP: {len(clientes_sp)}")

    # Encadeando múltiplos where() — funciona como AND
    stmt2 = (
        select(Cliente)
        .where(Cliente.estado == "SP")
        .where(Cliente.cidade == "São Paulo")
    )
    clientes_sao_paulo = session.scalars(stmt2).all()
    print(f"Clientes na cidade de São Paulo: {len(clientes_sao_paulo)}")

Clientes em SP: 4
Clientes na cidade de São Paulo: 2


### 3.2 Condições com OR

In [17]:
with Session(engine) as session:

    # Clientes de SP OU RJ
    stmt = select(Cliente).where(
        or_(Cliente.estado == "SP", Cliente.estado == "RJ")
    )
    # Alternativa mais pythônica para listas:
    # .where(Cliente.estado.in_(["SP", "RJ"]))

    clientes = session.scalars(stmt).all()
    print(f"Clientes em SP ou RJ: {len(clientes)}")

Clientes em SP ou RJ: 9


### 3.3 Busca com LIKE

In [18]:
with Session(engine) as session:

    # Produtos cujo nome contém "Note"
    stmt = select(Produto).where(Produto.nome.like("%Note%"))
    produtos = session.scalars(stmt).all()

    for p in produtos:
        print(f"  {p.nome} — R${p.preco_atual}")

    if not produtos:
        print("Nenhum produto encontrado (popule o banco primeiro)")

  Notebook Pro 15 — R$3499.90


### 3.4 Montando queries dinamicamente

No trabalho, muitas vezes os filtros são opcionais (vêm de um formulário, de uma API...).  
O padrão abaixo é muito comum em código de produção:

In [21]:
def buscar_produtos(session, categoria=None, preco_max=None, apenas_ativos=True):
    """Busca produtos com filtros opcionais — padrão comum em APIs."""
    
    stmt = select(Produto)  # começa sem filtros

    if apenas_ativos:
        stmt = stmt.where(Produto.ativo == True)

    if categoria:
        stmt = stmt.where(Produto.categoria == categoria)

    if preco_max is not None:
        stmt = stmt.where(Produto.preco_atual <= preco_max)

    return session.scalars(stmt).all()


with Session(engine) as session:
    # Diferentes combinações de filtros
    todos_ativos = buscar_produtos(session)
    eletronicos  = buscar_produtos(session, categoria="Eletrônicos")
    baratos      = buscar_produtos(session, preco_max=Decimal("1000"))

    print(f"Todos os ativos:     {len(todos_ativos)}")
    print(f"Eletrônicos:         {len(eletronicos)}")
    print(f"Até R$1.000,00:      {len(baratos)}")

Todos os ativos:     39
Eletrônicos:         6
Até R$1.000,00:      5


## 4. Ordenando resultados

In [23]:
with Session(engine) as session:  # Abre uma sessão para executar a query no banco

    # Produtos ordenados por preço (maior primeiro)
    stmt = (  # Monta a query usando select() com encadeamento de métodos
        select(Produto.nome, Produto.categoria, Produto.preco_atual)  # Seleciona nome, categoria e preço dos produtos
        .where(Produto.ativo == True)  # Filtra apenas produtos ativos (ativo = True)
        .order_by(Produto.preco_atual.desc())  # Ordena por preço decrescente (maior primeiro); use .asc() para crescente
    )

    produtos = session.execute(stmt).all()  # Executa a query e retorna todos os resultados como tuplas

    print(f"{'Produto':<30} {'Categoria':<20} {'Preço':>10}")  # Imprime o cabeçalho da tabela formatado
    print("-" * 62)  # Imprime uma linha separadora
    for p in produtos[:5]:  # Itera apenas nos primeiros 5 produtos para limitar o output
        print(f"{p.nome:<30} {p.categoria:<20} R${p.preco_atual:>8}")  # Imprime cada produto formatado (nome alinhado à esquerda, categoria, preço à direita)

Produto                        Categoria                 Preço
--------------------------------------------------------------
Mollitia ipsam                 Livros               R$ 4847.27
Ipsum nihil repudiandae expedita Roupas               R$ 4754.24
Qui magnam                     Esportes             R$ 4705.14
Aperiam voluptas veniam        Livros               R$ 4633.57
Vitae ipsum                    Eletrônicos          R$ 4171.79


## 5. Paginação com limit() e offset()

Essencial para APIs e dashboards que não podem carregar tudo de uma vez.

In [25]:
def listar_clientes_paginados(session, pagina: int = 20, por_pagina: int = 5):  # Define função para listar clientes paginados
    """Retorna clientes paginados, ordenados por nome.""" 
    
    offset = (pagina - 1) * por_pagina  # Calcula o offset baseado na página e itens por página

    stmt = (  # Monta a query SQLAlchemy
        select(Cliente)  # Seleciona a tabela Cliente
        .order_by(Cliente.nome)  # Ordena os resultados por nome
        .limit(por_pagina)  # Limita o número de resultados por página
        .offset(offset)  # Pula os registros anteriores à página atual
    )

    return session.scalars(stmt).all()  # Executa a query e retorna lista de objetos Cliente


with Session(engine) as session:  # Abre uma sessão para executar queries
    # Total de clientes para calcular o total de páginas
    total = session.scalar(select(func.count()).select_from(Cliente))  # Conta o total de clientes na tabela
    print(f"Total de clientes: {total}")  # Imprime o total de clientes

    pagina_1 = listar_clientes_paginados(session, pagina=1, por_pagina=5)  # Chama a função para a página 1
    print(f"\nPágina 1 (5 por página):")  # Imprime cabeçalho da página
    for c in pagina_1:  # Itera sobre os clientes da página
        print(f"  {c.nome} — {c.cidade}/{c.estado}")  # Imprime nome e localização de cada cliente

Total de clientes: 21

Página 1 (5 por página):
  Alícia Alves — Duque de Caxias/RJ
  Ana Liz Monteiro — Duque de Caxias/RJ
  Anna Liz Sá — Brasília/DF
  Davi Luiz Pimenta — Fortaleza/CE
  Deborah Foroni — São Paulo/SP


---

## Exercício: Usando IA para isso

Uma das melhores aplicações de IA no dia a dia de dados: **transformar perguntas em código de query**.

**Prompt para traduzir perguntas em queries:**
```
Usando os modelos SQLAlchemy abaixo, escreva uma query Python que responda:
"Quais são os 5 produtos mais vendidos do mês passado,
agrupados por categoria, com o total de receita de cada?"

[cole os modelos ORM aqui]
```
---

### Resposta:Código gerado pelo Copilot

In [26]:
from sqlalchemy import select, func
from datetime import datetime, timedelta

# Calcula o mês passado
hoje = datetime.now()
mes_passado = hoje.replace(day=1) - timedelta(days=1)
inicio_mes_passado = mes_passado.replace(day=1)
fim_mes_passado = mes_passado

with Session(engine) as session:
    # Query para os 5 produtos mais vendidos do mês passado, agrupados por categoria
    stmt = (
        select(
            Produto.categoria,  # Agrupa por categoria
            Produto.nome,  # Nome do produto
            func.sum(ItemPedido.quantidade).label("total_vendido"),  # Soma da quantidade vendida
            func.sum(ItemPedido.preco_venda * ItemPedido.quantidade).label("receita_total")  # Soma da receita (preço * quantidade)
        )
        .join(ItemPedido, ItemPedido.produto_id == Produto.id)  # JOIN com ItemPedido
        .join(Pedido, Pedido.id == ItemPedido.pedido_id)  # JOIN com Pedido
        .where(Pedido.data_pedido >= inicio_mes_passado)  # Filtra pedidos do mês passado
        .where(Pedido.data_pedido <= fim_mes_passado)
        .group_by(Produto.categoria, Produto.nome)  # Agrupa por categoria e nome do produto
        .order_by(func.sum(ItemPedido.quantidade).desc())  # Ordena por total vendido (desc)
        .limit(5)  # Limita aos 5 primeiros
    )

    resultados = session.execute(stmt).all()

    # Exibe os resultados
    print("5 Produtos Mais Vendidos do Mês Passado (por Categoria):")
    print(f"{'Categoria':<20} {'Produto':<30} {'Total Vendido':<15} {'Receita Total':>15}")
    print("-" * 80)
    for row in resultados:
        print(f"{row.categoria:<20} {row.nome:<30} {row.total_vendido:<15} R${row.receita_total:>14.2f}")

5 Produtos Mais Vendidos do Mês Passado (por Categoria):
Categoria            Produto                        Total Vendido     Receita Total
--------------------------------------------------------------------------------
Livros               Unde enim ea                   14              R$      18438.28
Livros               Iste vitae delectus            12              R$      56777.28
Roupas               Illum error                    12              R$       2011.68
Eletrônicos          Vitae ipsum                    10              R$      41717.90
Roupas               Doloremque sapiente            10              R$       6514.60
